# Compare scoring stages by epoch

This notebook compares stage labels between two scoring JSON files.

- It uses **only epochs present in** `81012_T0_scoring (1).json`.
- It compares each epoch's `stage` value against `comparison/81012_T0_scoring.json`.
- It reports agreement metrics and lists mismatched epochs.

In [6]:
import json
from collections import Counter, defaultdict
from pathlib import Path

short_path = Path("/Users/julian/Walkingframes Dropbox/Julian Amacker/PhD_2024/Code_external/ScoringHero-main/data/81012_T0_scoring (1).json")
full_path = Path("/Users/julian/Walkingframes Dropbox/Julian Amacker/PhD_2024/Code_external/ScoringHero-main/data/comparison/81012_T0_scoring.json")

short_path, full_path

(PosixPath('/Users/julian/Walkingframes Dropbox/Julian Amacker/PhD_2024/Code_external/ScoringHero-main/data/81012_T0_scoring (1).json'),
 PosixPath('/Users/julian/Walkingframes Dropbox/Julian Amacker/PhD_2024/Code_external/ScoringHero-main/data/comparison/81012_T0_scoring.json'))

In [7]:
def extract_epochs(payload):
    """Handle both [epochs, ...] and plain epochs list formats."""
    if isinstance(payload, list) and payload and isinstance(payload[0], list):
        return payload[0]
    if isinstance(payload, list):
        return payload
    raise ValueError("Unsupported JSON format: expected a list or [list, ...].")


def to_epoch_stage_map(path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    epochs = extract_epochs(data)

    mapping = {}
    for row in epochs:
        if "epoch" not in row or "stage" not in row:
            raise ValueError(f"Missing required fields in row: {row}")
        mapping[int(row["epoch"])] = row["stage"]

    return mapping


short_map = to_epoch_stage_map(short_path)
full_map = to_epoch_stage_map(full_path)

print(f"Epochs in short file: {len(short_map)}")
print(f"Epochs in comparison file: {len(full_map)}")

Epochs in short file: 1818
Epochs in comparison file: 1818


In [8]:
# Restrict comparison to epochs that are present in the short file.
# Exclude epochs where stage is None in either file.
rows = []
excluded_none = 0

for epoch in sorted(short_map.keys()):
    stage_short = short_map.get(epoch)
    stage_comparison = full_map.get(epoch)

    if stage_short is None or stage_comparison is None:
        excluded_none += 1
        continue

    rows.append(
        {
            "epoch": epoch,
            "stage_short": stage_short,
            "stage_comparison": stage_comparison,
            "stage_short_norm": stage_short,
            "stage_comparison_norm": stage_comparison,
            "match": stage_short == stage_comparison,
        }
    )

total = len(rows)
matches = sum(1 for r in rows if r["match"])
mismatches = total - matches
agreement_percent = round(100 * matches / total, 2) if total else None

summary = {
    "epochs_considered_from_short_file": len(short_map),
    "epochs_excluded_due_to_none_stage": excluded_none,
    "epochs_compared": total,
    "matches": matches,
    "mismatches": mismatches,
    "agreement_percent": agreement_percent,
}

summary

{'epochs_considered_from_short_file': 1818,
 'epochs_excluded_due_to_none_stage': 1306,
 'epochs_compared': 512,
 'matches': 488,
 'mismatches': 24,
 'agreement_percent': 95.31}

In [9]:
mismatches_list = [
    {
        "epoch": r["epoch"],
        "stage_short": r["stage_short"],
        "stage_comparison": r["stage_comparison"],
    }
    for r in rows
    if not r["match"]
]

print(f"Mismatched epochs: {len(mismatches_list)}")
print("First 50 mismatches:")

for item in mismatches_list[:50]:
    print(item)

Mismatched epochs: 24
First 50 mismatches:
{'epoch': 208, 'stage_short': 'N2', 'stage_comparison': 'Wake'}
{'epoch': 209, 'stage_short': 'N2', 'stage_comparison': 'Wake'}
{'epoch': 210, 'stage_short': 'N2', 'stage_comparison': 'Wake'}
{'epoch': 211, 'stage_short': 'N2', 'stage_comparison': 'N1'}
{'epoch': 212, 'stage_short': 'Wake', 'stage_comparison': 'N1'}
{'epoch': 219, 'stage_short': 'N1', 'stage_comparison': 'N2'}
{'epoch': 220, 'stage_short': 'N1', 'stage_comparison': 'N2'}
{'epoch': 232, 'stage_short': 'N2', 'stage_comparison': 'N3'}
{'epoch': 234, 'stage_short': 'N2', 'stage_comparison': 'N3'}
{'epoch': 235, 'stage_short': 'N2', 'stage_comparison': 'N3'}
{'epoch': 236, 'stage_short': 'N2', 'stage_comparison': 'N3'}
{'epoch': 330, 'stage_short': 'N2', 'stage_comparison': 'REM'}
{'epoch': 332, 'stage_short': 'N1', 'stage_comparison': 'REM'}
{'epoch': 333, 'stage_short': 'N1', 'stage_comparison': 'REM'}
{'epoch': 359, 'stage_short': 'REM', 'stage_comparison': 'N2'}
{'epoch': 360, 

In [10]:
# Optional: confusion matrix for a quick overview.
confusion = defaultdict(Counter)

for r in rows:
    confusion[r["stage_short_norm"]][r["stage_comparison_norm"]] += 1

all_cols = sorted({col for row_counts in confusion.values() for col in row_counts.keys()})
print("Confusion matrix (counts):")
print("short_file\\comparison_file", *all_cols, sep="\t")

for short_stage in sorted(confusion.keys()):
    counts = [str(confusion[short_stage].get(col, 0)) for col in all_cols]
    print(short_stage, *counts, sep="\t")

Confusion matrix (counts):
short_file\comparison_file	N1	N2	N3	REM	Wake
N1	6	3	0	4	0
N2	1	29	10	1	3
N3	0	0	206	0	0
REM	0	1	0	40	0
Wake	1	0	0	0	207
